# 07 - Station Detail Cards

This notebook generates per-station performance cards and JSON data.

**Features:**
- Individual station metrics
- Mini trend visualizations
- JSON exports for infographic section

**Output:** `static/data/stations/*.json`

In [1]:
import os
import sys
import json
import sqlite3
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

project_root = os.path.abspath(os.path.join(os.getcwd(), '..','..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

DB_PATH = os.path.join(project_root, 'data', 'caltrain_lat_long.db')
GTFS_PATH = os.path.join(project_root, 'gtfs_data')
METADATA_PATH = os.path.join(project_root, 'data', 'station_metadata.json')
OUTPUT_DIR = os.path.join(project_root, 'static', 'data', 'stations')

print(f"Database exists: {os.path.exists(DB_PATH)}")
print(f"Metadata exists: {os.path.exists(METADATA_PATH)}")

Database exists: True
Metadata exists: True


## Configuration

In [2]:
# ====================
# CONFIGURATION
# ====================

DAYS_TO_ANALYZE = 30

## Load Station Metadata

In [3]:
def load_metadata():
    with open(METADATA_PATH, 'r') as f:
        data = json.load(f)
    
    # Create stop_id to station mapping
    stop_to_station = {}
    for s in data['stations']:
        for stop_id in s['stop_ids']:
            stop_to_station[stop_id] = s['id']
    
    return data['stations'], stop_to_station

stations, stop_to_station = load_metadata()
print(f"Loaded {len(stations)} stations")

Loaded 31 stations


## Load and Process Data

In [4]:
sys.path.insert(0, os.path.join(project_root, 'src'))
from utils.time_utils import calculate_time_difference, normalize_time
from utils.geo_utils import haversine

def load_arrivals():
    conn = sqlite3.connect(DB_PATH)
    cutoff = (datetime.now() - timedelta(days=DAYS_TO_ANALYZE)).strftime('%Y-%m-%d')
    query = f"SELECT trip_id, stop_id, vehicle_lat, vehicle_lon, timestamp FROM train_locations WHERE timestamp >= '{cutoff}'"
    raw_df = pd.read_sql_query(query, conn)
    conn.close()
    
    if raw_df.empty:
        return pd.DataFrame()
    
    raw_df['timestamp'] = pd.to_datetime(raw_df['timestamp'], errors='coerce')
    raw_df = raw_df.dropna(subset=['timestamp'])
    
    stops_df = pd.read_csv(os.path.join(GTFS_PATH, 'stops.txt'))
    stops_df = stops_df[stops_df['stop_id'].str.isnumeric()]
    stop_times_df = pd.read_csv(os.path.join(GTFS_PATH, 'stop_times.txt'))
    
    raw_df['stop_id'] = pd.to_numeric(raw_df['stop_id'].astype(str), errors='coerce').astype('Int64')
    raw_df['trip_id'] = pd.to_numeric(raw_df['trip_id'].astype(str), errors='coerce').astype('Int64')
    stops_df['stop_id'] = pd.to_numeric(stops_df['stop_id'].astype(str), errors='coerce').astype('Int64')
    stop_times_df['stop_id'] = pd.to_numeric(stop_times_df['stop_id'].astype(str), errors='coerce').astype('Int64')
    stop_times_df['trip_id'] = pd.to_numeric(stop_times_df['trip_id'].astype(str), errors='coerce').astype('Int64')
    raw_df = raw_df.dropna(subset=['trip_id', 'stop_id'])
    
    df = pd.merge(raw_df, stop_times_df[['trip_id', 'stop_id', 'arrival_time']], on=['trip_id', 'stop_id'], how='inner')
    df = pd.merge(df, stops_df[['stop_id', 'stop_lat', 'stop_lon']], on='stop_id', how='inner')
    
    df['distance'] = df.apply(lambda r: haversine(r['vehicle_lat'], r['vehicle_lon'], r['stop_lat'], r['stop_lon']), axis=1)
    df['date'] = df['timestamp'].dt.date
    df['hour'] = df['timestamp'].dt.hour
    df['arrival_time'] = df['arrival_time'].apply(normalize_time)
    
    min_dist = df.groupby(['trip_id', 'stop_id', 'date'])['distance'].min().reset_index()
    merged = pd.merge(df, min_dist, on=['trip_id', 'stop_id', 'date', 'distance'])
    arrivals = merged.groupby(['trip_id', 'stop_id', 'date']).first().reset_index()
    
    arrivals['delay_min'] = arrivals.apply(lambda r: calculate_time_difference(r['arrival_time'], r['timestamp']), axis=1)
    arrivals.loc[arrivals.delay_min > 500, 'delay_min'] = 0
    arrivals.loc[arrivals.delay_min < -100, 'delay_min'] = 0
    
    arrivals['on_time'] = arrivals['delay_min'] <= 4
    arrivals['status'] = 'On Time'
    arrivals.loc[(arrivals.delay_min > 4) & (arrivals.delay_min <= 15), 'status'] = 'Minor'
    arrivals.loc[arrivals.delay_min > 15, 'status'] = 'Major'
    
    # Map to parent station
    arrivals['station_id'] = arrivals['stop_id'].apply(lambda x: stop_to_station.get(str(x)))
    arrivals = arrivals[arrivals['station_id'].notna()]
    
    print(f"Processed {len(arrivals):,} arrivals")
    return arrivals

arrivals = load_arrivals()

Base directory: D:\caltrain-tracker
Dotenv path: D:\caltrain-tracker\.env
Dotenv file exists: True
Processed 48,518 arrivals


## Calculate Per-Station Metrics

In [5]:
def calculate_station_detail(arrivals, station_id):
    """Calculate detailed metrics for a single station."""
    station_data = arrivals[arrivals['station_id'] == station_id]
    
    if station_data.empty:
        return None
    
    # Basic metrics
    metrics = {
        'total_arrivals': int(len(station_data)),
        'on_time_pct': round((station_data['on_time'].mean() * 100), 1),
        'avg_delay': round(station_data['delay_min'].mean(), 1),
        'max_delay': round(station_data['delay_min'].max(), 1),
    }
    
    # Status breakdown
    metrics['status_breakdown'] = {
        'on_time': int((station_data['status'] == 'On Time').sum()),
        'minor': int((station_data['status'] == 'Minor').sum()),
        'major': int((station_data['status'] == 'Major').sum()),
    }
    
    # Busiest hours
    hour_counts = station_data.groupby('hour').size()
    if not hour_counts.empty:
        busiest_hour = int(hour_counts.idxmax())
        metrics['busiest_hour'] = busiest_hour
        metrics['busiest_hour_name'] = f"{busiest_hour}:00 - {busiest_hour+1}:00"
    
    # Daily trend (last 7 days)
    daily = station_data.groupby('date')['on_time'].mean().reset_index()
    daily['date'] = pd.to_datetime(daily['date'])
    daily = daily.sort_values('date').tail(7)
    metrics['daily_trend'] = [
        {'date': str(row['date'].date()), 'value': round(row['on_time'] * 100, 1)}
        for _, row in daily.iterrows()
    ]
    
    # Performance rating
    pct = metrics['on_time_pct']
    if pct >= 90:
        metrics['rating'] = 'excellent'
    elif pct >= 80:
        metrics['rating'] = 'good'
    elif pct >= 70:
        metrics['rating'] = 'fair'
    else:
        metrics['rating'] = 'poor'
    
    return metrics

# Test with one station
test_metrics = calculate_station_detail(arrivals, 'san_francisco')
if test_metrics:
    print(json.dumps(test_metrics, indent=2))

{
  "total_arrivals": 1179,
  "on_time_pct": 100.0,
  "avg_delay": -41.7,
  "max_delay": 0.0,
  "status_breakdown": {
    "on_time": 1179,
    "minor": 0,
    "major": 0
  },
  "busiest_hour": 8,
  "busiest_hour_name": "8:00 - 9:00",
  "daily_trend": [
    {
      "date": "2025-12-22",
      "value": 100.0
    },
    {
      "date": "2025-12-23",
      "value": 100.0
    },
    {
      "date": "2025-12-24",
      "value": 100.0
    },
    {
      "date": "2025-12-25",
      "value": 100.0
    },
    {
      "date": "2025-12-26",
      "value": 100.0
    },
    {
      "date": "2025-12-27",
      "value": 100.0
    },
    {
      "date": "2025-12-28",
      "value": 100.0
    }
  ],
  "rating": "excellent"
}


## Generate All Station Files

In [6]:
def generate_all_stations(arrivals, stations):
    """Generate JSON files for all stations."""
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    generated = []
    
    for station in stations:
        station_id = station['id']
        metrics = calculate_station_detail(arrivals, station_id)
        
        if metrics is None:
            print(f"⚠️ No data for {station['name']}")
            continue
        
        # Combine with metadata
        output = {
            'id': station_id,
            'name': station['name'],
            'order': station['order'],
            'lat': station['lat'],
            'lon': station['lon'],
            'image_prompt': station.get('image_prompt', ''),
            'metrics': metrics,
            'generated_at': datetime.now().isoformat(),
        }
        
        # Save to file
        filepath = os.path.join(OUTPUT_DIR, f"{station_id}.json")
        with open(filepath, 'w') as f:
            json.dump(output, f, indent=2)
        
        generated.append(station_id)
        print(f"✅ {station['name']:25} {metrics['on_time_pct']:5.1f}% ({metrics['total_arrivals']:,} arrivals)")
    
    return generated

print("Generating station files...\n")
generated = generate_all_stations(arrivals, stations)
print(f"\n✅ Generated {len(generated)} station files")

Generating station files...

✅ San Francisco             100.0% (1,179 arrivals)
✅ 22nd Street                90.4% (2,467 arrivals)
✅ Bayshore                   90.1% (1,945 arrivals)
✅ South San Francisco        91.4% (2,468 arrivals)
✅ San Bruno                  92.5% (1,945 arrivals)
✅ Millbrae                   92.8% (2,458 arrivals)
✅ Broadway                   91.2% (559 arrivals)
✅ Burlingame                 92.8% (1,945 arrivals)
✅ San Mateo                  92.7% (2,458 arrivals)
✅ Hayward Park               93.3% (1,946 arrivals)
✅ Hillsdale                  92.6% (2,456 arrivals)
✅ Belmont                    94.2% (1,904 arrivals)
✅ San Carlos                 91.7% (1,915 arrivals)
✅ Redwood City               93.6% (2,430 arrivals)
⚠️ No data for Stanford
✅ Menlo Park                 93.1% (2,206 arrivals)
✅ Palo Alto                  92.0% (2,466 arrivals)
✅ California Avenue          91.1% (2,212 arrivals)
✅ San Antonio                92.8% (2,207 arrivals)
✅ Mountain Vi

## Generate Station Index

In [7]:
def generate_index(arrivals, stations):
    """Generate index file with all stations summary."""
    summary = []
    
    for station in stations:
        station_id = station['id']
        metrics = calculate_station_detail(arrivals, station_id)
        
        if metrics:
            summary.append({
                'id': station_id,
                'name': station['name'],
                'order': station['order'],
                'on_time_pct': metrics['on_time_pct'],
                'total_arrivals': metrics['total_arrivals'],
                'rating': metrics['rating'],
            })
    
    # Sort by geographic order
    summary.sort(key=lambda x: x['order'])
    
    index = {
        'generated_at': datetime.now().isoformat(),
        'days_analyzed': DAYS_TO_ANALYZE,
        'station_count': len(summary),
        'stations': summary,
    }
    
    filepath = os.path.join(OUTPUT_DIR, '_index.json')
    with open(filepath, 'w') as f:
        json.dump(index, f, indent=2)
    
    print(f"✅ Station index: {filepath}")
    return index

index = generate_index(arrivals, stations)

✅ Station index: d:\caltrain-tracker\static\data\stations\_index.json


## Summary

In [8]:
print("=" * 60)
print("🚉 STATION DATA GENERATION COMPLETE")
print("=" * 60)

print(f"\n📁 Output directory: {OUTPUT_DIR}")
print(f"📊 Stations generated: {len(generated)}")
print(f"📅 Days analyzed: {DAYS_TO_ANALYZE}")

# Top and bottom performers
sorted_stations = sorted(index['stations'], key=lambda x: x['on_time_pct'], reverse=True)

print("\n🏆 Top 5 Stations:")
for s in sorted_stations[:5]:
    print(f"  {s['name']:25} {s['on_time_pct']:5.1f}%")

print("\n⚠️ Bottom 5 Stations:")
for s in sorted_stations[-5:]:
    print(f"  {s['name']:25} {s['on_time_pct']:5.1f}%")

🚉 STATION DATA GENERATION COMPLETE

📁 Output directory: d:\caltrain-tracker\static\data\stations
📊 Stations generated: 29
📅 Days analyzed: 30

🏆 Top 5 Stations:
  San Francisco             100.0%
  San Jose Diridon          100.0%
  Belmont                    94.2%
  San Martin                 93.7%
  Redwood City               93.6%

⚠️ Bottom 5 Stations:
  College Park               90.8%
  22nd Street                90.4%
  Bayshore                   90.1%
  Capitol                    89.5%
  Tamien                     82.6%
